# Reusable Template: Screening & Relative-Scoring Pipeline

A parameterized, function-based version of the lab you can adapt to **any**
"acquire → clean → benchmark/score → filter → visualize" project — not just
stocks. Examples: screening real-estate listings against neighborhood
averages, ranking product SKUs against category benchmarks, shortlisting
job candidates against role benchmarks, etc.

**How to reuse this**: edit the `CONFIG` cell, point `load_data()` at your
real source, and the rest of the pipeline runs unchanged.


## 1. Configuration — edit this for your project

In [1]:
CONFIG = {
    # Column that uniquely identifies each row (e.g. Ticker, SKU, Listing ID)
    "id_col": "Ticker",

    # Columns that define the peer-group hierarchy, coarse -> fine
    # (e.g. ["Sector"], or ["Sector", "Industry"], or ["City", "Neighborhood"])
    "group_cols": ["Sector", "Industry"],

    # Columns stored as messy text that need %, $, and comma stripping
    "percent_or_plain_numeric_cols": [
        "P/E", "Forward P/E", "PEG", "P/S", "P/B", "P/Cash", "EPS (ttm)",
        "EPS growth next year", "EPS growth next 5 years",
        "Total Debt/Equity", "Beta", "Institutional Ownership", "Price", "Volume",
    ],

    # Columns needing a custom unit parser -> function name
    "special_numeric_cols": {"Market Cap": "clean_market_cap"},

    # Metrics to benchmark against peer-group averages ("lower is better" here;
    # flip the comparison in `flag_undervalued` if higher-is-better for your project)
    "benchmark_metrics": ["Price", "P/E", "PEG", "P/S", "P/B"],

    # Minimum composite score to call something a "prospect"
    "score_threshold": 8,

    # Final multi-criteria screen: list of (column, "min"/"max"/"eq", value)
    "screen_criteria": [
        ("Price", "min", 20), ("Price", "max", 100),
        ("Volume", "min", 10_000),
        ("Country", "eq", "USA"),
        ("EPS (ttm)", "min", 0),
        ("EPS growth next year", "min", 0),
        ("EPS growth next 5 years", "min", 0),
        ("Total Debt/Equity", "max", 1),
        ("Beta", "max", 1.5),
        ("Institutional Ownership", "max", 30),
    ],

    "max_shortlist": 6,
}


## 2. Load data

Swap this function's body for your real data source (CSV, API, database query).

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def load_data(source="synthetic", path=None, n=400, seed=42):
    """Load the raw cross-sectional dataset.
    source="synthetic": use the bundled generator (offline, reproducible demo data).
    source="csv": load a real export from `path` with pd.read_csv.
    """
    if source == "synthetic":
        from data_utils import make_finviz_like_raw
        return make_finviz_like_raw(n=n, seed=seed)
    elif source == "csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unknown source: {source}")

raw = load_data(source="synthetic")
raw.head()


,No.,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Forward P/E,PEG,...,P/B,P/Cash,EPS (ttm),EPS growth next year,EPS growth next 5 years,Total Debt/Equity,Beta,Institutional Ownership,Price,Volume
0,1,WCS,Berkshire Hathaway Inc.,Financial,Property & Casualty Insurance,USA,1.39B,15.40,14.28,-,...,6.62,21.77,1.67,19.27%,15.61%,0.23,1.18,31.0%,"172,000.00","1,434,753"
1,2,XRU,XRU Holdings Inc.,Services,Auto Parts Stores,UK,4.57B,27.48,26.10,2.65,...,2.08,5.99,-0.53,14.16%,23.55%,0.45,0.58,34.4%,25.79,"1,091,512"
2,3,VLUV,VLUV Holdings Inc.,Industrial Goods,Industrial Equipment & Components,USA,120.17M,25.35,23.56,2.10,...,5.30,2.33,0.70,3.30%,2.33%,0.39,1.75,33.4%,44.29,"1,598,054"
3,4,COUH,COUH Holdings Inc.,Financial,Asset Management,USA,555.76M,34.87,24.65,0.10,...,1.31,16.44,1.86,14.90%,4.87%,0.58,1.31,47.3%,31.34,"865,007"
4,5,MSEX,MSEX Holdings Inc.,Basic Materials,Gold,USA,4.53B,31.98,26.70,1.37,...,0.66,2.35,-1.81,-1.97%,14.80%,0.22,0.81,87.5%,42.00,"326,063"


## 3. Clean numeric columns (generic)

In [3]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Strip %, $, commas; treat '-' as missing; cast to float."""
    cleaned = (series.astype(str)
                      .str.replace("%", "", regex=False)
                      .str.replace("$", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .str.strip()
                      .replace("-", np.nan))
    return pd.to_numeric(cleaned, errors="coerce")

def clean_market_cap(series: pd.Series) -> pd.Series:
    """Example custom unit parser: 'B'/'M' suffixed strings -> $ millions float."""
    def _conv(x):
        x = str(x).strip()
        if x.endswith("B"): return float(x[:-1]) * 1000
        if x.endswith("M"): return float(x[:-1])
        try: return float(x)
        except ValueError: return np.nan
    return series.apply(_conv)

SPECIAL_PARSERS = {"clean_market_cap": clean_market_cap}

def clean_dataframe(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    df = df.copy()
    for col in config["percent_or_plain_numeric_cols"]:
        if col in df.columns:
            df[col] = clean_numeric(df[col])
    for col, parser_name in config.get("special_numeric_cols", {}).items():
        if col in df.columns:
            df[col] = SPECIAL_PARSERS[parser_name](df[col])
    return df

clean = clean_dataframe(raw, CONFIG)
clean.dtypes


No.                          int64
Ticker                         str
Company                        str
Sector                         str
Industry                       str
Country                        str
Market Cap                 float64
P/E                        float64
Forward P/E                float64
PEG                        float64
P/S                        float64
P/B                        float64
P/Cash                     float64
EPS (ttm)                  float64
EPS growth next year       float64
EPS growth next 5 years    float64
Total Debt/Equity          float64
Beta                       float64
Institutional Ownership    float64
Price                      float64
Volume                       int64
dtype: object

## 4. Outlier check (generic, on the first benchmark metric)

In [4]:
def find_extreme_outlier(df: pd.DataFrame, metric: str, group_cols: list):
    """Drill from the top group average down to the single most extreme row.
    Returns the id value of that row (or None if nothing looks extreme)."""
    level0 = df.groupby(group_cols[0])[metric].mean().sort_values(ascending=False)
    top_group = level0.index[0]
    sub = df[df[group_cols[0]] == top_group]
    if len(group_cols) > 1:
        level1 = sub.groupby(group_cols[1])[metric].mean().sort_values(ascending=False)
        top_subgroup = level1.index[0]
        sub = sub[sub[group_cols[1]] == top_subgroup]
    return sub.loc[sub[metric].idxmax(), CONFIG["id_col"]]

outlier_id = find_extreme_outlier(clean, CONFIG["benchmark_metrics"][0], CONFIG["group_cols"])
print("Flagged outlier:", outlier_id)

clean_no_outlier = clean[clean[CONFIG["id_col"]] != outlier_id].reset_index(drop=True)


Flagged outlier: WCS


## 5. Peer-group benchmarks + composite score (generic)

In [5]:
def compute_group_benchmarks(df: pd.DataFrame, group_cols: list, metrics: list, prefix: str):
    return (df.groupby(group_cols)[metrics].mean()
              .add_prefix(prefix).reset_index())

def attach_benchmarks(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    out = df.copy()
    for depth in range(1, len(config["group_cols"]) + 1):
        cols = config["group_cols"][:depth]
        prefix = "".join(c[0].upper() for c in cols) + "Avg_"
        bench = compute_group_benchmarks(out, cols, config["benchmark_metrics"], prefix)
        out = out.merge(bench, on=cols)
    return out

def flag_undervalued(df: pd.DataFrame, config: dict, lower_is_better: bool = True) -> pd.DataFrame:
    out = df.copy()
    flag_cols = []
    bench_prefixes = [c for c in out.columns if c.endswith("Avg_" + config["benchmark_metrics"][0])]
    prefixes = sorted({c.split(config["benchmark_metrics"][0])[0] for c in out.columns
                        if c.endswith(config["benchmark_metrics"][0]) and c != config["benchmark_metrics"][0]})
    for m in config["benchmark_metrics"]:
        for p in prefixes:
            bench_col = f"{p}{m}"
            if bench_col in out.columns:
                flag_col = f"{p}{m}_Under"
                out[flag_col] = ((out[m] < out[bench_col]) if lower_is_better
                                  else (out[m] > out[bench_col])).astype(int)
                flag_cols.append(flag_col)
    out["Score"] = out[flag_cols].sum(axis=1)
    return out

with_bench = attach_benchmarks(clean_no_outlier, CONFIG)
scored = flag_undervalued(with_bench, CONFIG)
scored[[CONFIG["id_col"], "Score"]].sort_values("Score", ascending=False).head()


,Ticker,Score
4,LXD,10
356,NKK,10
354,RWXA,10
353,BGIT,10
334,HZL,10


## 6. Generic multi-criteria screen

In [6]:
def apply_screen(df: pd.DataFrame, criteria: list, score_col: str, score_threshold: float) -> pd.DataFrame:
    mask = pd.Series(True, index=df.index)
    for col, kind, value in criteria:
        if kind == "min":
            mask &= df[col] > value
        elif kind == "max":
            mask &= df[col] < value
        elif kind == "eq":
            mask &= df[col] == value
        else:
            raise ValueError(f"Unknown criterion kind: {kind}")
    mask &= df[score_col] >= score_threshold
    return df[mask].sort_values(score_col, ascending=False)

shortlist = apply_screen(scored, CONFIG["screen_criteria"], "Score", CONFIG["score_threshold"])
print(len(shortlist))
shortlist[[CONFIG["id_col"], "Score"]].head(CONFIG["max_shortlist"])


3


,Ticker,Score
41,GDB,10
111,CBW,10
371,YILO,8


## 7. Time-series add-on: moving averages for a shortlist (optional)

In [7]:
def add_moving_averages(history: pd.DataFrame, value_col: str, group_col: str, windows=(50, 200)):
    out = history.copy()
    for w in windows:
        out[f"MA{w}"] = out.groupby(group_col)[value_col].transform(lambda s: s.rolling(w).mean())
    return out

def plot_group_series(history: pd.DataFrame, group_col: str, x_col: str, y_col: str, extra_cols=()):
    for key, sub in history.groupby(group_col):
        fig, ax = plt.subplots()
        ax.plot(sub[x_col], sub[y_col], label=y_col, alpha=0.6)
        for c in extra_cols:
            ax.plot(sub[x_col], sub[c], label=c)
        ax.set_title(f"{key}")
        ax.legend()
        plt.show()

# Example usage once you have a real/simulated time series keyed by the shortlist ids:
# from data_utils import simulate_price_history
# ids = shortlist[CONFIG["id_col"]].head(CONFIG["max_shortlist"]).tolist()
# hist = simulate_price_history(ids)
# hist = add_moving_averages(hist, "AdjClose", "Symbol")
# plot_group_series(hist, "Symbol", "Date", "AdjClose", extra_cols=["MA50", "MA200"])


## 8. Adapting this template to a new project — checklist

1. Point `load_data()` at your real source (CSV export, API call, DB query).
2. Update `CONFIG["id_col"]` and `CONFIG["group_cols"]` to your entity id and
   peer-group hierarchy (coarse → fine).
3. List every messy numeric text column in `percent_or_plain_numeric_cols`;
   add a custom parser function + entry in `special_numeric_cols` for any
   column with non-standard units (like Market Cap's B/M suffixes here).
4. Choose `benchmark_metrics` — the columns you want compared to peer-group
   averages — and decide if lower-is-better or higher-is-better for your
   domain (pass `lower_is_better=False` to `flag_undervalued` if needed;
   for mixed-direction metrics, extend the function to take per-metric
   direction).
5. Set `screen_criteria` to your hard business rules and `score_threshold`
   to how many benchmark comparisons a candidate must "win."
6. If there's a time dimension, reuse `add_moving_averages` /
   `plot_group_series` with your own time-series data.
